# MaroMart Intent Understanding - ViT5 Training (Sync with Temo/search)
Notebook này đã được cấu hình để khớp với cấu trúc thư mục Google Drive của bạn:
- **Kết quả:** Lưu vào `Temo/search/file_train` 
- **Dữ liệu:** Đọc từ `Temo/search/dataset/semantic_dataset.csv`

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DRIVE_PATH = '/content/drive/MyDrive/Temo/search'
SAVE_PATH = f'{BASE_DRIVE_PATH}/file_train'
DATASET_DIR = f'{BASE_DRIVE_PATH}/dataset'

# Kiểm tra các thư mục
for path in [SAVE_PATH, DATASET_DIR]:
    if not os.path.exists(path):
        os.makedirs(path)
        print(f'Đã tạo thư mục: {path}')
    else:
        print(f'Thư mục sẵn sàng: {path}')

In [ ]:
# 2. Cài đặt thư viện
!pip install -q transformers sentencepiece torch pandas scikit-learn tqdm

In [ ]:
# 3. Import thư viện
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration, AutoTokenizer, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import json
import numpy as np
from tqdm.auto import tqdm

### 4. Đọc Dữ liệu
AI sẽ tìm file `semantic_dataset.csv` trong thư mục `Temo/search/dataset` trên Drive của bạn.

In [ ]:
dataset_file = f'{DATASET_DIR}/semantic_dataset.csv'

if not os.path.exists(dataset_file):
    print(f"⚠️ Không tìm thấy file tại {dataset_file}. Vui lòng upload file vào thư mục dataset trên Drive.")
    # Nếu không có, cho phép upload thủ công vào session tạm
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(list(uploaded.keys())[0])
else:
    df = pd.read_csv(dataset_file)
    print(f"✅ Đã tải dữ liệu thành công từ Drive ({len(df)} dòng).")

In [ ]:
class MaroMartDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input_len=256, max_target_len=512):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        input_text = str(row['input'])
        target_text = str(row['product_title']) + " - " + str(row['product_description'])

        inputs = self.tokenizer.encode_plus(
            input_text,
            max_length=self.max_input_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        targets = self.tokenizer.encode_plus(
            target_text,
            max_length=self.max_target_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": inputs["input_ids"].flatten(),
            "attention_mask": inputs["attention_mask"].flatten(),
            "labels": targets["input_ids"].flatten()
        }

In [ ]:
def calculate_metrics(preds, labels):
    preds = preds.flatten()
    labels = labels.flatten()
    mask = labels != 0
    preds = preds[mask]
    labels = labels[mask]
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    return acc, precision, recall, f1

def train_and_evaluate(config, train_df, val_df, device):
    model_name = "VietAI/vit5-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = T5ForConditionalGeneration.from_pretrained(model_name)
    model.config.dropout_rate = config['dropout']
    model.to(device)

    train_dataset = MaroMartDataset(train_df, tokenizer)
    val_dataset = MaroMartDataset(val_df, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'])

    optimizer = AdamW(model.parameters(), lr=config['lr'])
    epochs = 3
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            outputs.loss.backward()
            optimizer.step()

    model.eval()
    def get_scores(loader):
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in loader:
                input_ids = batch["input_ids"].to(device)
                labels = batch["labels"].to(device)
                outputs = model.generate(input_ids=input_ids, max_length=512)
                padded_outputs = torch.zeros_like(labels)
                limit = min(outputs.shape[1], labels.shape[1])
                padded_outputs[:, :limit] = outputs[:, :limit]
                all_preds.append(padded_outputs.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        return calculate_metrics(np.concatenate(all_preds), np.concatenate(all_labels))

    train_scores = get_scores(train_loader)
    val_scores = get_scores(val_loader)
    
    # Lưu mô hình của config này
    # model.save_pretrained(f"{SAVE_PATH}/model_d{config['dropout']}_lr{config['lr']}")
    
    return train_scores, val_scores

In [ ]:
# 5. Chạy Grid Search và Lưu trực tiếp vào Temo/search/file_train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
dropouts, batch_sizes, lrs = [0.1, 0.3], [8, 16], [1e-5, 3e-5, 5e-5] # Rút gọn bớt để chạy nhanh hơn
results = []

for d in dropouts:
    for b in batch_sizes:
        for lr in lrs:
            config = {'dropout': d, 'batch_size': b, 'lr': lr}
            print(f"--- Đang huấn luyện: {config} ---")
            t_s, v_s = train_and_evaluate(config, train_df, val_df, device)
            
            res = {
                "config": config,
                "train": {"acc": t_s[0], "pre": t_s[1], "rec": t_s[2], "f1": t_s[3]},
                "val": {"acc": v_s[0], "pre": v_s[1], "rec": v_s[2], "f1": v_s[3]}
            }
            results.append(res)
            
            # Lưu file JSON vào Drive sau mỗi vòng lặp
            with open(f'{SAVE_PATH}/vit5_training_results.json', 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=4)
            print(f"✅ Đã cập nhật kết quả vào: {SAVE_PATH}")

print(f"\n🔥 TẤT CẢ ĐÃ HOÀN TẤT! Toàn bộ file đã nằm trong Google Drive của bạn.")